<a href="https://colab.research.google.com/github/Yash-Agarwal-4a5h/cei-assignments-yash-agarwal-jecrc-college/blob/main/week7_yash_agarwal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Document Question Answering System (RAG)

This notebook builds a simple Retrieval-Augmented Generation system.

It will:
- Load PDF documents
- Split them into chunks
- Convert chunks into embeddings
- Store embeddings in a vector database
- Retrieve relevant chunks for a question
- Generate an answer from the retrieved context

In [ ]:
%pip install -q pypdf sentence-transformers faiss-cpu transformers accelerate datasets

Note: you may need to restart the kernel to use updated packages.


## 1. Import Libraries

We need tools for reading PDFs, chunking text, creating embeddings, searching with FAISS, and generating answers with a language model.

In [ ]:
import os
import glob
import numpy as np
import faiss
import torch

from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

## 2. Load Documents

Place your PDFs inside a folder named `documents`.

If you do not have PDFs, you can use any text document or later switch to the Hugging Face dataset.

In [ ]:
def extract_text_from_pdf(pdf_path):
    reader = PdfReader(pdf_path)
    text = ""
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text + "\n"
    return text

documents_folder = "documents"
pdf_files = glob.glob(os.path.join(documents_folder, "*.pdf"))

all_texts = []
for pdf in pdf_files:
    text = extract_text_from_pdf(pdf)
    if text.strip():
        all_texts.append({
            "source": os.path.basename(pdf),
            "text": text
        })

print("PDF files found:", len(pdf_files))
print("Loaded documents:", len(all_texts))

PDF files found: 1
Loaded documents: 1


## 3. Split Text Into Chunks

Long documents are split into smaller pieces so retrieval works better.

In [ ]:
def split_text(text, chunk_size=800, overlap=150):
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        start = end - overlap
        if start < 0:
            start = 0
        if start >= len(text):
            break

    return chunks

chunks = []
metadata = []

for doc in all_texts:
    doc_chunks = split_text(doc["text"])
    for i, chunk in enumerate(doc_chunks):
        chunks.append(chunk)
        metadata.append({
            "source": doc["source"],
            "chunk_id": i
        })

print("Total chunks:", len(chunks))

Total chunks: 7


## 4. Create Embeddings

Each chunk is converted into a vector using a sentence embedding model.

In [ ]:
if not chunks:
    raise ValueError("No text chunks found. Add at least one PDF to the documents folder.")

embedder = SentenceTransformer("all-MiniLM-L6-v2")
chunk_embeddings = embedder.encode(chunks, convert_to_numpy=True, show_progress_bar=True)

if chunk_embeddings.ndim == 1:
    chunk_embeddings = np.expand_dims(chunk_embeddings, axis=0)

print("Embedding shape:", chunk_embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding shape: (7, 384)


## 5. Build the Vector Database

We use FAISS to store embeddings and search for the most relevant chunks.

In [ ]:
print("pdf_files:", len(pdf_files))
print("all_texts:", len(all_texts))
print("chunks:", len(chunks))
print("chunk_embeddings.shape:", chunk_embeddings.shape)

pdf_files: 1
all_texts: 1
chunks: 7
chunk_embeddings.shape: (7, 384)


In [ ]:
dimension = chunk_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(chunk_embeddings.astype(np.float32))

print("Number of vectors in index:", index.ntotal)

Number of vectors in index: 7


## 6. Define Retrieval Function

This function takes a user question, finds the most similar document chunks, and returns them as context.

In [ ]:
def retrieve_context(question, top_k=3):
    query_embedding = embedder.encode([question], convert_to_numpy=True)
    if query_embedding.ndim == 1:
        query_embedding = np.expand_dims(query_embedding, axis=0)

    distances, indices = index.search(query_embedding.astype(np.float32), top_k)

    retrieved = []
    for idx in indices[0]:
        if idx != -1:
            retrieved.append({
                "text": chunks[idx],
                "metadata": metadata[idx]
            })
    return retrieved

## 7. Load the Language Model

We use a small text-to-text model to generate the final answer from the retrieved context.

In [ ]:
model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


## 8. Answer Generation Function

The prompt includes the retrieved context so the answer stays grounded in the document.

In [ ]:
def generate_answer(prompt, max_new_tokens=256):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).to(device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
def answer_question(question, top_k=3):
    retrieved = retrieve_context(question, top_k=top_k)

    if not retrieved:
        return "No relevant context found.", []

    context = "\n\n".join(
        [f"Source: {item['metadata']['source']}\nChunk: {item['text']}" for item in retrieved]
    )

    prompt = f"""
Use the context below to answer the question.
If the answer is not in the context, say you do not know.

Context:
{context}

Question: {question}

Answer:
"""

    result = generate_answer(prompt)
    return result, retrieved

## 9. Ask a Sample Question

Now we test the system with a simple question.

In [ ]:
question = "What is the main idea of the document?"
answer, sources = answer_question(question)

print("Question:", question)
print("\nAnswer:\n", answer)
print("\nRetrieved Sources:")
for s in sources:
    print("-", s["metadata"]["source"], "| Chunk", s["metadata"]["chunk_id"])

Question: What is the main idea of the document?

Answer:
 reducing validation gap by 20–25% through regularization techniques and AI-driven summarization workflows, reducing manual review effort by 40%. Research Paper Recommendation System [LINK ], (Python, NLP, Sentence Transformers, MLP, Streamlit) •Processed and analyzed 5000+ research paper records and Top-5 recommendations with 90% relevence accuracy. •Optimized recommendation workflows and performance evaluation processes, improving recommendation precision by 20% and reducing manual search effort by 40%. Plagiarism Detection System [LINK ], (Python, NLP, TF-IDF, Logistic Regression, Flask, SQLite) •Analyzed 2000+ text samples using NLP preprocessing techniques, achieving 90% similarity detection accuracy. •Optimized preprocessing workflows to reduce false similarity matches by 15% and built SQL-supported pipelines. SKILLS •Programming Languages: Python (intermediate), SQL (intermediate) •Core CS Concept

Retrieved Sources:
- Ga

## 10. Try More Questions

You can test different questions and see which chunks are retrieved.

In [ ]:
questions = [
    "What is the document about?",
    "What are the key findings?",
    "What is the conclusion?"
]

for q in questions:
    ans, _ = answer_question(q)
    print("Q:", q)
    print("A:", ans)
    print("-" * 80)

Q: What is the document about?
A: Gaurang_Gupta_29th_June (2).pdf
--------------------------------------------------------------------------------
Q: What are the key findings?
A: •Developing expertise in cloud engineering, distributed systems, and scalable technology architectures and using GenAI. •Building proficiency in cybersecurity principles and regulatory frameworks for secure digital systems for secure landscapes. •Bridging technical development and business strategy through cloud-based CRM and enterprise solutions. •Developed an AI-powered Bank Loan Recommendation
--------------------------------------------------------------------------------
Q: What is the conclusion?
A: Developing expertise in cloud engineering, distributed systems, and scalable technology architectures and using GenAI. •Building proficiency in cybersecurity principles and regulatory frameworks for secure digital systems for secure landscapes. •Bridging technical development and business strategy through cl

## 11. Summary

This notebook is fully local:
- no API keys
- no Pinecone
- no Cohere
- no Streamlit required

To use it, just add PDFs to the `documents` folder and rerun the cells.